# Indusense — Sprint 1

Ce notebook produira les cartes d'identité des fichiers CSV placés dans `datas/`.

In [ ]:
import csv
from codecs import BOM_UTF8
from collections.abc import Iterator
from datetime import date, datetime, time
from io import StringIO
from pathlib import Path

import pandas as pd
from sqlglot import exp, parse
from sqlglot.errors import ParseError, TokenError

DATA_DIR = Path("datas")

## Lire les fichiers CSV

On conserve les valeurs brutes sous forme de texte pour préserver les identifiants. Les champs vides sont considérés comme manquants. L'encodage est déduit en essayant UTF-8, puis Windows-1252 et ISO-8859-1.

In [ ]:
def lire_csv(chemin: Path) -> tuple[pd.DataFrame, str, str]:
    contenu = chemin.read_bytes()
    encodages = ("utf-8-sig",) if contenu.startswith(BOM_UTF8) else (
        "utf-8", "cp1252", "iso-8859-1"
    )
    for encodage in encodages:
        try:
            texte = contenu.decode(encodage)
        except UnicodeDecodeError:
            continue
        break
    else:
        raise ValueError(f"{chemin}: le fichier ne peut pas être décodé.")

    try:
        separateur = csv.Sniffer().sniff(texte[:8192], delimiters=",;\t").delimiter
    except csv.Error as erreur:
        raise ValueError(f"{chemin}: séparateur CSV indéterminé.") from erreur

    try:
        donnees = pd.read_csv(
            StringIO(texte),
            sep=separateur,
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )
    except (pd.errors.ParserError, pd.errors.EmptyDataError) as erreur:
        raise ValueError(f"{chemin}: lecture du CSV impossible ({erreur}).") from erreur

    return donnees, encodage, separateur

## Inférer les types et calculer les statistiques

Les colonnes connues suivent les règles explicites du projet. Pour les autres, toutes les valeurs présentes doivent être compatibles avec le type inféré. Les dates et heures inférées suivent les formats ISO. Les horodatages avec fuseau sont comparés en UTC. Une colonne mixte reste en texte et la carte indique la famille compatible avec le plus de valeurs. Les champs vides sont exclus de l’inférence et des statistiques.

In [ ]:
IDENTIFIANTS = {"incident_id", "machine_id", "operator_badge"}
NUMERIQUES = {
    "releves_incidents.csv": {"severity"},
    "telemetry.csv": {
        "temperature_c", "pressure_bar", "voltage_mean_v",
        "rotation_mean_rpm", "pieces_produced",
    },
}
FORMATS_TEMPORELS = {
    "releves_incidents.csv": {
        "date": ("DATE", "%Y-%m-%d"),
        "time": ("TIME", "%H:%M"),
    },
    "telemetry.csv": {"timestamp": ("DATETIME", "%Y-%m-%d %H:%M:%S")},
}
COLONNES_INDICATEURS = {
    "type_surchauffe", "type_baisse_pression", "type_vibration",
    "type_bruit_mecanique", "type_surconsommation", "type_blocage_mecanique",
    "type_alarme_capteur", "type_arret_urgence", "type_defaut_qualite",
}
INDICATEURS = {"0": False, "1": True}
DATE_ISO = r"[0-9]{4}-[0-9]{2}-[0-9]{2}"
HEURE_ISO = r"(?:[01][0-9]|2[0-3]):[0-5][0-9](?::[0-5][0-9](?:\.[0-9]{1,6})?)?"
DATETIME_ISO = rf"{DATE_ISO}[T ]{HEURE_ISO}"
FUSEAU_ISO = r"(?:Z|[+-](?:[01][0-9]|2[0-3]):?[0-5][0-9])"


def convertir_numeriques(valeurs: pd.Series) -> pd.Series:
    """Écarte aussi les valeurs infinies des statistiques."""
    nombres = pd.to_numeric(valeurs, errors="coerce")
    return nombres.mask(nombres.isin([float("inf"), float("-inf")]))


def candidats_temporels(valeurs: pd.Series) -> Iterator[tuple[str, pd.Series]]:
    """Ne propose que des formats ISO non ambigus, sans mélanger les horloges."""
    for type_colonne, motif, avec_fuseau in (
        ("DATE", DATE_ISO, False),
        ("TIME", HEURE_ISO, False),
        ("DATETIME", DATETIME_ISO, False),
        ("DATETIME (avec fuseau)", DATETIME_ISO + FUSEAU_ISO, True),
    ):
        correspondantes = valeurs[valeurs.str.fullmatch(motif)]
        if correspondantes.empty:
            continue
        if type_colonne == "TIME":
            converties = correspondantes.map(time.fromisoformat)
        else:
            converties = pd.to_datetime(
                correspondantes, format="ISO8601", errors="coerce", utc=avec_fuseau
            )
        yield type_colonne, converties


def inferer_type(
    fichier: str, nom: str, valeurs: pd.Series, format_temporel: tuple[str, str] | None
) -> tuple[str, pd.Series, str, str]:
    """Renvoie le type, les valeurs converties, la provenance et le diagnostic."""
    explicite = "Règle explicite"
    observe = "Valeurs observées"
    nom_normalise = nom.casefold()
    if nom_normalise in IDENTIFIANTS or nom_normalise.endswith(("_id", "_badge")):
        return "IDENTITY", valeurs, explicite, ""
    if fichier == "releves_incidents.csv" and nom in COLONNES_INDICATEURS:
        return "BOOLEAN", valeurs.map(INDICATEURS), explicite, ""
    if format_temporel is not None:
        type_colonne, format_date = format_temporel
        dates = pd.to_datetime(valeurs, format=format_date, errors="coerce")
        return type_colonne, dates, explicite, ""
    if nom in NUMERIQUES.get(fichier, ()):
        return "NUMERIC", convertir_numeriques(valeurs), explicite, ""
    if valeurs.empty:
        return "TEXT", valeurs, observe, "Aucune valeur non vide"

    if valeurs.str.fullmatch(r"[0-9]+").all() and valeurs.str.match(r"0[0-9]").any():
        return "TEXT", valeurs, observe, "Zéros initiaux conservés"

    candidats = [("NUMERIC", convertir_numeriques(valeurs)), *candidats_temporels(valeurs)]
    type_probable, converties = max(candidats, key=lambda candidat: candidat[1].count())
    compatibles = converties.count()
    if compatibles == len(valeurs):
        return type_probable.partition(" ")[0], converties, observe, ""
    if compatibles:
        diagnostic = (
            f"Valeurs mixtes : {compatibles}/{len(valeurs)} compatibles avec {type_probable}"
        )
        return "TEXT", valeurs, observe, diagnostic
    return "TEXT", valeurs, observe, ""


def calculer_statistiques(
    type_colonne: str, valeurs: pd.Series, *, format_date: str | None = None
) -> dict[str, int | float | str]:
    if valeurs.empty:
        return {}
    if type_colonne == "NUMERIC":
        return {
            "min": valeurs.min().item(),
            "max": valeurs.max().item(),
            "mean": valeurs.mean().item(),
            "median": valeurs.median().item(),
        }
    if type_colonne in {"DATE", "TIME", "DATETIME"}:
        bornes = {"min": valeurs.min(), "max": valeurs.max()}
        if format_date is not None:
            return {cle: valeur.strftime(format_date) for cle, valeur in bornes.items()}
        if type_colonne == "DATE":
            return {cle: valeur.date().isoformat() for cle, valeur in bornes.items()}
        return {cle: valeur.isoformat() for cle, valeur in bornes.items()}
    return {}

## Décrire les deux fichiers

Chaque description regroupe les métadonnées, le type des colonnes et les statistiques. Les horaires sont ceux de la machine qui exécute le notebook. La section suivante utilise ces descriptions pour générer les cartes Markdown.

In [ ]:
def decrire_csv(chemin: Path) -> dict[str, object]:
    debut = datetime.now().astimezone()
    source = chemin.resolve()
    taille = source.stat().st_size
    donnees, encodage, separateur = lire_csv(source)
    colonnes = []

    for nom, serie in donnees.items():
        # Un champ composé seulement d'espaces compte aussi comme manquant.
        valeurs = serie[serie.notna() & serie.str.strip().ne("")]
        format_temporel = FORMATS_TEMPORELS.get(source.name, {}).get(nom)
        type_colonne, converties, provenance, diagnostic = inferer_type(
            source.name, nom, valeurs, format_temporel
        )
        invalides = int(converties.isna().sum())
        if invalides:
            raise ValueError(
                f"{source.name}, colonne {nom}: {invalides} valeur(s) "
                f"invalide(s) pour le type {type_colonne}."
            )
        colonnes.append({
            "nom": nom,
            "type": type_colonne,
            "provenance": provenance,
            "diagnostic": diagnostic,
            "valeurs_manquantes": len(serie) - len(valeurs),
            "statistiques": calculer_statistiques(
                type_colonne, converties, format_date=format_temporel[1] if format_temporel else None
            ),
        })

    lignes, nombre_colonnes = donnees.shape
    return {
        "source": str(source),
        "taille_octets": taille,
        "debut": debut,
        "fin": datetime.now().astimezone(),
        "encodage": encodage,
        "separateur": separateur,
        "lignes": lignes,
        "nombre_colonnes": nombre_colonnes,
        "colonnes": colonnes,
    }

In [ ]:
FICHIERS_CSV = ("releves_incidents.csv", "telemetry.csv")
descriptions = {nom: decrire_csv(DATA_DIR / nom) for nom in FICHIERS_CSV}

pd.DataFrame([
    {
        "Fichier": nom,
        "Encodage": description["encodage"],
        "Séparateur": repr(description["separateur"]),
        "Lignes": description["lignes"],
        "Colonnes": description["nombre_colonnes"],
    }
    for nom, description in descriptions.items()
])

## Générer les cartes d'identité Markdown

Chaque carte prend le nom du fichier source, la date et l'heure de début (AAAAMMJJ_HHMMSS) de son analyse. Les rapports sont enregistrés dans le dossier daté sous `artifacts/ingestion/identities/`.

In [ ]:
IDENTITIES_DIR = Path("artifacts/ingestion/identities")


def formater_taille(octets: int) -> str:
    taille = float(octets)
    for unite in ("octets", "Ko", "Mo", "Go", "To"):
        if taille < 1_000 or unite == "To":
            return f"{taille:.2f} {unite}"
        taille /= 1_000


def rendre_entete_markdown(description: dict[str, object]) -> list[str]:
    source = Path(description["source"])
    debut = description["debut"]
    fin = description["fin"]
    return [
        f"# Carte d'identité — {source.name}",
        "",
        f"- Source: `{source}`",
        f"- Filsize: {formater_taille(description['taille_octets'])}",
        f"- ProcessBeginAt: {debut:%H:%M:%S}",
        f"- ProcessEndAt: {fin:%H:%M:%S}",
        f"- Charset: {description['encodage']}",
        f"- Rows Number: {description['lignes']}",
        f"- Cols Number: {description['nombre_colonnes']}",
    ]


def rendre_tableaux_colonnes(colonnes: list[dict], *, afficher_type_sql: bool = False) -> list[str]:
    niveau = "###" if afficher_type_sql else "##"
    lignes = [
        "",
        f"{niveau} Column list",
        "",
    ]
    if afficher_type_sql:
        lignes.extend([
            "| Colname | SQL Type | Inferred Type | Raison du classement |",
            "| --- | --- | --- | --- |",
        ])
    else:
        lignes.extend([
            "| Colname | Inferred Type | Origine | Diagnostic |",
            "| --- | --- | --- | --- |",
        ])
    statistiques = []
    for colonne in colonnes:
        nom = colonne["nom"].replace("\\", "\\\\").replace("|", "\\|")
        type_colonne = colonne["type"]
        if afficher_type_sql:
            lignes.append(
                f"| {nom} | {colonne['type_sql']} | {type_colonne} | {colonne['raison']} |"
            )
        else:
            provenance = colonne["provenance"]
            diagnostic = colonne["diagnostic"] or "—"
            lignes.append(f"| {nom} | {type_colonne} | {provenance} | {diagnostic} |")
        if type_colonne in {"NUMERIC", "DATE", "TIME", "DATETIME"}:
            valeurs = colonne["statistiques"]
            cellules = [str(valeurs.get(cle, "—")) for cle in ("min", "max", "mean", "median")]
            statistiques.append("| " + " | ".join([nom, *cellules]) + " |")

    if statistiques:
        lignes.extend([
            "",
            f"{niveau} Statistiques",
            "",
            "| Colname | Min value | Max value | Mean value | Median value |",
            "| --- | --- | --- | --- | --- |",
            *statistiques,
        ])
    return lignes


def rendre_carte_markdown(description: dict[str, object]) -> str:
    lignes = rendre_entete_markdown(description) + rendre_tableaux_colonnes(description["colonnes"])
    return "\n".join(lignes) + "\n"


def ecrire_carte_markdown(
    description: dict[str, object], destination: Path, contenu: str
) -> Path:
    """Enregistre une carte sans écraser un rapport existant."""
    debut = description["debut"]
    dossier = destination / f"{debut:%Y%m%d}"
    nom_source = Path(description["source"]).stem
    chemin = dossier / f"{nom_source}_{debut:%Y%m%d_%H%M%S}.md"
    dossier.mkdir(parents=True, exist_ok=True)
    with chemin.open("x", encoding="utf-8", newline="\n") as fichier:
        fichier.write(contenu)
    return chemin

In [ ]:
cartes_markdown = {
    nom: ecrire_carte_markdown(description, IDENTITIES_DIR, rendre_carte_markdown(description))
    for nom, description in descriptions.items()
}

for chemin in cartes_markdown.values():
    print(chemin.as_posix())

## Décrire le script PostgreSQL

SQLGlot lit les définitions de tables et les lignes explicites des `INSERT … VALUES`. La carte indique si le classement vient d'une clé primaire, d'une référence, de la convention de nommage des identifiants ou du type déclaré. Les statistiques décrivent les valeurs présentes dans le script. Les valeurs par défaut calculées par PostgreSQL et les effets des clauses `ON CONFLICT` dépendent de l'exécution en base.

In [ ]:
DIALECTE_SQL = "postgres"
TYPES_SQL_NUMERIQUES = {"SMALLINT", "INT", "BIGINT", "DECIMAL", "FLOAT", "DOUBLE"}


def extraire_tables_sql(instructions: list[exp.Expr]) -> dict:
    tables = {}
    for instruction in instructions:
        if not isinstance(instruction, exp.Create) or instruction.kind != "TABLE":
            continue
        schema = instruction.this
        if not isinstance(schema, exp.Schema):
            raise ValueError(f"Colonnes non déclarées pour la table {schema.name}.")
        nom_table = schema.this.sql(dialect=DIALECTE_SQL)
        tables[nom_table] = {
            "definitions": {
                colonne.name: colonne for colonne in schema.expressions
                if isinstance(colonne, exp.ColumnDef)
            },
            "lignes": [],
        }

    for instruction in instructions:
        if not isinstance(instruction, exp.Insert):
            continue
        cible = instruction.this
        table = cible.this if isinstance(cible, exp.Schema) else cible
        nom_table = table.sql(dialect=DIALECTE_SQL)
        if nom_table not in tables or not isinstance(instruction.expression, exp.Values):
            raise ValueError(f"INSERT non pris en charge pour {nom_table}.")
        donnees = tables[nom_table]
        definitions = donnees["definitions"]
        noms = [colonne.name for colonne in cible.expressions] if isinstance(
            cible, exp.Schema
        ) else list(definitions)
        if len(noms) != len(set(noms)) or not set(noms) <= definitions.keys():
            raise ValueError(f"Colonnes INSERT invalides pour {nom_table}.")
        for ligne in instruction.expression.expressions:
            if len(ligne.expressions) != len(noms):
                raise ValueError(f"Nombre de valeurs INSERT incorrect pour {nom_table}.")
            donnees["lignes"].append(dict(zip(noms, ligne.expressions, strict=True)))
    return tables


def classer_colonne_sql(colonne: exp.ColumnDef) -> tuple[str, str]:
    contraintes = tuple(contrainte.kind for contrainte in colonne.constraints)
    if any(isinstance(contrainte, exp.PrimaryKeyColumnConstraint) for contrainte in contraintes):
        return "IDENTITY", "Clé primaire"
    if any(isinstance(contrainte, exp.Reference) for contrainte in contraintes):
        return "IDENTITY", "Référence"
    if colonne.name.endswith("_id"):
        return "IDENTITY", "Convention de nommage (_id)"

    type_sql = colonne.kind.this.name
    if type_sql in TYPES_SQL_NUMERIQUES:
        return "NUMERIC", "Type déclaré"
    type_colonne = {
        "BOOLEAN": "BOOLEAN", "DATE": "DATE", "TIME": "TIME",
        "TIMESTAMP": "DATETIME", "TIMESTAMPTZ": "DATETIME",
    }.get(type_sql, "TEXT")
    return type_colonne, "Type déclaré"


def profil_colonne_sql(colonne: exp.ColumnDef, lignes: list[dict]) -> dict:
    nom = colonne.name
    type_colonne, raison = classer_colonne_sql(colonne)
    valeurs = []
    for ligne in lignes:
        valeur = ligne.get(nom)
        if valeur is None or isinstance(valeur, exp.Null):
            continue
        if isinstance(valeur, exp.Literal):
            valeurs.append(valeur.this)
        elif isinstance(valeur, exp.Neg) and isinstance(valeur.this, exp.Literal):
            valeurs.append(f"-{valeur.this.this}")
        elif type_colonne in {"NUMERIC", "DATE", "TIME", "DATETIME"}:
            raise ValueError(f"Valeur SQL non littérale dans la colonne {nom}.")

    statistiques = {}
    if valeurs and type_colonne == "NUMERIC":
        nombres = convertir_numeriques(pd.Series(valeurs, dtype="string"))
        if nombres.isna().any():
            raise ValueError(f"Valeur numérique SQL invalide dans la colonne {nom}.")
        statistiques = calculer_statistiques(type_colonne, nombres)
    elif valeurs and type_colonne in {"DATE", "TIME", "DATETIME"}:
        if type_colonne == "DATE":
            dates = [date.fromisoformat(valeur) for valeur in valeurs]
        elif type_colonne == "TIME":
            dates = [time.fromisoformat(valeur) for valeur in valeurs]
        else:
            dates = [datetime.fromisoformat(valeur) for valeur in valeurs]
        statistiques = {"min": min(dates).isoformat(), "max": max(dates).isoformat()}

    return {
        "nom": nom,
        "type": type_colonne,
        "type_sql": colonne.kind.sql(dialect=DIALECTE_SQL),
        "raison": raison,
        "statistiques": statistiques,
    }


def decrire_sql(chemin: Path) -> dict:
    debut = datetime.now().astimezone()
    source = chemin.resolve()
    contenu = source.read_bytes()
    encodage = "utf-8-sig" if contenu.startswith(BOM_UTF8) else "utf-8"
    try:
        instructions = [
            instruction for instruction in parse(contenu.decode(encodage), read=DIALECTE_SQL)
            if instruction is not None
        ]
        tables = extraire_tables_sql(instructions)
        profils = [
            {
                "nom": nom,
                "lignes": len(table["lignes"]),
                "colonnes": [
                    profil_colonne_sql(colonne, table["lignes"])
                    for colonne in table["definitions"].values()
                ],
            }
            for nom, table in tables.items()
        ]
    except (ValueError, TokenError, ParseError) as erreur:
        raise ValueError(f"{source.name}: {erreur}") from erreur
    return {
        "source": str(source),
        "taille_octets": len(contenu),
        "debut": debut,
        "fin": datetime.now().astimezone(),
        "encodage": encodage,
        "instructions": len(instructions),
        "tables": profils,
        "lignes": sum(table["lignes"] for table in profils),
        "nombre_colonnes": sum(len(table["colonnes"]) for table in profils),
    }

In [ ]:
def rendre_carte_sql(description: dict) -> str:
    lignes = rendre_entete_markdown(description)
    lignes.extend([
        "- Dialecte: PostgreSQL",
        f"- Instructions: {description['instructions']}",
        f"- Tables: {len(description['tables'])}",
        "",
        "Les lignes comptées correspondent aux tuples des INSERT. Les statistiques "
        "portent sur les valeurs explicites. Le symbole — indique une statistique "
        "indisponible ou sans objet, notamment pour les champs alimentés par DEFAULT NOW().",
    ])
    for table in description["tables"]:
        lignes.extend([
            "",
            f"## Table `{table['nom']}`",
            "",
            f"- Rows Number: {table['lignes']}",
            f"- Cols Number: {len(table['colonnes'])}",
            *rendre_tableaux_colonnes(table["colonnes"], afficher_type_sql=True),
        ])
    return "\n".join(lignes) + "\n"


description_sql = decrire_sql(DATA_DIR / "machine.sql.txt")
carte_sql = ecrire_carte_markdown(
    description_sql, IDENTITIES_DIR, rendre_carte_sql(description_sql)
)
print(carte_sql.as_posix())